# 01 — Classificatore a feature (TF-IDF + regressione logistica)

Sul dataset finale (`1-data-cleaning/03_dataset_finale`): solo il **testo degli articoli**.

Fold **per gruppo** (la valutazione principale: tutte le testate di un gruppo, cioè rete o stesso owner, nello
stesso fold) e **per testata** (confronto: i gruppi possono essere spezzati). Un modello per fold: vocabolario e
IDF si stimano **solo sul training** del fold, il test passa solo da `transform`. Iperparametri fissi.

Con `TAGLIO_TOKEN` il testo si taglia ai primi N token del tokenizer di ModernBERT: lo stesso testo che vede il
transformer con `MAX_TOKEN` = N.

Accanto al classificatore due baseline senza testo: la **classe di maggioranza** del training e una **scelta a
caso** con le proporzioni del training.

Si salva **tutto**, per poter ricalcolare qualsiasi metrica senza riaddestrare:

- per ogni articolo del test: `article_id`, `domain`, `fold`, `y_true`, i **logit** `logit_l1..logit_l5`, le
  **probabilità** `p_l1..p_l5`, `y_pred`;
- `metriche.json`: per articolo, per testata col voto e per testata con la media delle probabilità —
  accuratezza, F1 macro e pesata, precisione/recall/F1 per livello, matrice di confusione, tolerant 0.5 e 1,
  errore medio in livelli, binario (soglia $l_2$/$l_3$) con TP, TN, FP, FN, precisione, recall, F1.

File: `Results/2-classificazione/<NOME>_<schema>_articoli.csv`, `<NOME>_<schema>_metriche.json`,
`<NOME>_metriche.csv` (tabella riassuntiva), `<NOME>_FATTO` quando tutto è finito.

**Sul server:** `cd ~/John/WebScience2027/2-classificazione && papermill -k mbvenv 01_classificatore_feature.ipynb ../Results/2-classificazione/esecuzioni/01_classificatore_feature.ipynb`
(di solito lo lancia `coda_weekend.py`).

In [1]:
import json
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, metriche, per_testata, registratore

In [2]:
NOME = "feature"
TAGLIO_TOKEN = 0           # 0 = testo intero; N = primi N token (tokenizer di ModernBERT)
C_GRIGLIA = ""             # "" = C fisso a 1; "0.1,1,10,100" = C scelto con una validazione interna per gruppi (solo training)
SCHEMI_DA_FARE = SCHEMI
N_JOBS = 5                 # fold in parallelo
PROVA = False              # True: 40 testate, risultati in Results/2-classificazione/prova/

In [3]:
# Parameters
NOME = "feature_tuned"
C_GRIGLIA = "0.1,1,10,100"


In [4]:
OUT = cartella_risultati(PROVA)
registra = registratore(OUT / f"avanzamento_{NOME}.log")
art = carica(PROVA)
if TAGLIO_TOKEN:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    enc = tok(art.text.tolist(), truncation=True, max_length=TAGLIO_TOKEN, add_special_tokens=False, return_offsets_mapping=True)
    art["text"] = [t[:o[-1][1]] if o else t for t, o in zip(art.text, enc["offset_mapping"])]
registra(f"=== {NOME}: {len(art)} articoli, {art.domain.nunique()} testate, taglio {TAGLIO_TOKEN or 'nessuno'} ===")

2026-10-18 13:52:10  === feature_tuned: 44029 articoli, 304 testate, taglio nessuno ===


## Addestramento

Per ogni fold: `fit_transform` sul training, `transform` sul test. I logit sono la `decision_function` della
regressione logistica.

In [5]:
def nuovo_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                           max_features=100_000, strip_accents="unicode")


def scegli_C(tr):
    """C con la F1 macro per testata (voto) più alta in una validazione a 3 fold per gruppi, sul solo training."""
    griglia = [float(x) for x in C_GRIGLIA.split(",")]
    testate = tr.drop_duplicates("domain").reset_index(drop=True)
    punteggi = {c: [] for c in griglia}
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(testate, testate.label, testate.gruppo):
        a, b = tr[tr.domain.isin(testate.domain[itr])], tr[tr.domain.isin(testate.domain[iva])]
        v = nuovo_tfidf(); Xa, Xb = v.fit_transform(a.text), v.transform(b.text)
        for c in griglia:
            p = LogisticRegression(C=c, max_iter=1000, class_weight="balanced", random_state=SEED).fit(Xa, a.label).predict(Xb)
            t = per_testata(pd.DataFrame({"domain": b.domain.values, "y_true": b.label.values, "y_pred": p}))
            punteggi[c].append(f1_score(t.y_true, t.y_pred, average="macro", labels=range(5), zero_division=0))
    medie = {c: float(np.mean(s)) for c, s in punteggi.items()}
    return max(medie, key=medie.get), medie


def un_fold(schema, fold):
    t0 = time.time()
    tr, te = art[art[schema] != fold], art[art[schema] == fold]
    C, medie = (1.0, {}) if not C_GRIGLIA else scegli_C(tr)
    if C_GRIGLIA:
        registra(f"{schema} fold {fold}: C scelto {C} (F1 interne {', '.join(f'{k}: {v:.3f}' for k, v in medie.items())})")
    v = nuovo_tfidf()
    clf = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", random_state=SEED)
    clf.fit(v.fit_transform(tr.text), tr.label)              # vocabolario, IDF e modello: solo training
    X = v.transform(te.text)                                   # test: solo transform
    registra(f"{schema} fold {fold}: fatto in {time.time() - t0:.0f} s ({len(tr)} training, {len(te)} test)")
    return te.index.to_numpy(), clf.classes_, clf.decision_function(X), clf.predict_proba(X)


def addestra(schema):
    file_out = OUT / f"{NOME}_{schema}_articoli.csv"
    if file_out.exists():
        registra(f"{schema}: già fatto, carico {file_out.name}")
        return pd.read_csv(file_out)
    logit, proba = np.full((len(art), 5), np.nan), np.zeros((len(art), 5))
    for idx, classi, z, p in Parallel(n_jobs=N_JOBS)(delayed(un_fold)(schema, f) for f in sorted(art[schema].unique())):
        logit[idx[:, None], classi[None, :]] = z
        proba[idx[:, None], classi[None, :]] = p
    pred = pd.DataFrame({"article_id": art.article_id, "domain": art.domain, "fold": art[schema], "y_true": art.label})
    for k in range(5):
        pred[f"logit_l{k + 1}"] = logit[:, k]
    for k, c in enumerate(CLASSI):
        pred[c] = proba[:, k]
    pred["y_pred"] = proba.argmax(1)
    pred.to_csv(file_out, index=False)
    json.dump(metriche(pred), open(OUT / f"{NOME}_{schema}_metriche.json", "w"), indent=1)
    return pred


pred = {s: addestra(s) for s in SCHEMI_DA_FARE}

2026-10-18 14:04:17  fold_gruppo fold 2: C scelto 0.1 (F1 interne 0.1: 0.336, 1.0: 0.334, 10.0: 0.327, 100.0: 0.311)


2026-10-18 14:04:19  fold_gruppo fold 5: C scelto 0.1 (F1 interne 0.1: 0.351, 1.0: 0.326, 10.0: 0.314, 100.0: 0.319)


2026-10-18 14:05:02  fold_gruppo fold 3: C scelto 0.1 (F1 interne 0.1: 0.352, 1.0: 0.328, 10.0: 0.343, 100.0: 0.340)


2026-10-18 14:06:06  fold_gruppo fold 4: C scelto 0.1 (F1 interne 0.1: 0.349, 1.0: 0.324, 10.0: 0.315, 100.0: 0.315)


2026-10-18 14:06:13  fold_gruppo fold 5: fatto in 838 s (35761 training, 8268 test)


...

2026-10-18 14:25:28  fold_testata fold 5: C scelto 1.0 (F1 interne 0.1: 0.293, 1.0: 0.307, 10.0: 0.296, 100.0: 0.281)


2026-10-18 14:25:46  fold_testata fold 1: fatto in 905 s (36256 training, 7773 test)


2026-10-18 14:26:44  fold_testata fold 3: fatto in 962 s (34879 training, 9150 test)


2026-10-18 14:26:59  fold_testata fold 4: fatto in 976 s (34790 training, 9239 test)


2026-10-18 14:28:10  fold_testata fold 5: fatto in 1046 s (35010 training, 9019 test)


## Baseline senza testo

Per ogni fold, dalle testate di training: la classe più frequente, e una classe estratta a caso con le
proporzioni del training (una per testata, seed fisso).

In [6]:
def baseline(schema, modo):
    rng = np.random.default_rng(SEED)
    y = np.zeros(len(art), dtype=int)
    for fold in sorted(art[schema].unique()):
        tr, te = art[art[schema] != fold], art[art[schema] == fold]
        priori = tr.drop_duplicates("domain").label.value_counts(normalize=True).sort_index()
        if modo == "maggioranza":
            y[te.index] = priori.idxmax()
        else:
            scelte = dict(zip(te.domain.unique(), rng.choice(priori.index, te.domain.nunique(), p=priori.values)))
            y[te.index] = te.domain.map(scelte)
    return pd.DataFrame({"domain": art.domain, "y_true": art.label, "y_pred": y})


for s in SCHEMI_DA_FARE:
    for modo in ["maggioranza", "a caso"]:
        json.dump(metriche(baseline(s, modo)), open(OUT / f"baseline_{modo.replace(' ', '_')}_{s}_metriche.json", "w"), indent=1)

## Risultati

Per articolo e per testata (voto di maggioranza): accuratezza a 5 classi, F1 macro, binario e tolerant.

In [7]:
righe = []
for s in SCHEMI_DA_FARE:
    righe.append({"modello": f"TF-IDF + LR ({NOME})", "fold": NOME_SCHEMA[s], **letture(pred[s])})
    for modo in ["maggioranza", "a caso"]:
        righe.append({"modello": f"baseline: {modo}", "fold": NOME_SCHEMA[s], **letture(baseline(s, modo))})
ris = pd.DataFrame(righe).set_index(["fold", "modello"])
ris.to_csv(OUT / f"{NOME}_metriche.csv")
ris[[c for c in ris.columns if c.startswith("testata")] + ["articolo 5cl", "articolo bin"]]

testata 5cl  testata F1  testata bin  \
fold        modello                                                             
per gruppo  TF-IDF + LR (feature_tuned)        0.424       0.380        0.740   
            baseline: maggioranza              0.316       0.096        0.553   
            baseline: a caso                   0.243       0.214        0.510   
per testata TF-IDF + LR (feature_tuned)        0.507       0.454        0.766   
            baseline: maggioranza              0.316       0.096        0.553   
            baseline: a caso                   0.234       0.209        0.523   

                                         testata tol0.5  testata tol1  \
fold        modello                                                     
per gruppo  TF-IDF + LR (feature_tuned)           0.597         0.770   
            baseline: maggioranza                 0.434         0.553   
            baseline: a caso                      0.388         0.533   
per testata TF-IDF + LR (feature_tuned)           0.660         0.812   
            baseline: maggioranza                 0.434         0.553   
            baseline: a caso                      0.382         0.530   

                                         articolo 5cl  articolo bin  
fold        modello                                                  
per gruppo  TF-IDF + LR (feature_tuned)         0.323         0.666  
            baseline: maggioranza               0.314         0.556  
            baseline: a caso                    0.245         0.523  
per testata TF-IDF + LR (feature_tuned)         0.421         0.716  
            baseline: maggioranza               0.314         0.556  
            baseline: a caso                    0.234         0.534

In [8]:
t = per_testata(pred[SCHEMI_DA_FARE[0]])
LIV = ["l1", "l2", "l3", "l4", "l5"]
display(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]))
(OUT / f"{NOME}_FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

predetto,l1,l2,l3,l4,l5
vero,,,,,
l1,46,13,3,8,1
l2,16,32,4,11,2
l3,7,8,5,15,1
l4,4,21,14,36,21
l5,3,7,2,14,10


19